# Homework 4: Flow past a body and drag

An ideal fluid (inviscid, incompressible) obeys the Euler equations:
$$ \begin{cases}
\rho\left(\frac{\partial\mathbf{u}}{\partial t} + \mathbf{u}\cdot\nabla\mathbf{u}\right) = -\nabla p, \\
\nabla\cdot\mathbf{u} = 0.
\end{cases} $$
To solve this in a simple fashion, we assume the flow is irrotational, $\nabla\times\mathbf{u} = 0$.
Then $\mathbf{u} = \nabla\phi$ for some scalar potential $\phi$. (Strictly, our domain has a hole
in it, so $\mathbf{u}$ could also circulate around the body; we take that circulation to be zero.)
Incompressibility, $\nabla\cdot\mathbf{u} = 0$, then gives Laplace's equation $\Delta\phi = 0$.
So the velocity comes out of a Laplace problem: solve for $\phi$, then take $\mathbf{u} = \nabla\phi$.

In [1]:
# On Google Colab, this installs FEniCSx, gmsh and viskex (takes a minute or two). Locally it does nothing.
import os
try:
    import dolfinx
except ImportError:
    os.system('wget "https://fem-on-colab.github.io/releases/fenicsx-install-release-real.sh" -O /tmp/fenicsx-install.sh && bash /tmp/fenicsx-install.sh')
    os.system('wget "https://fem-on-colab.github.io/releases/gmsh-install.sh" -O /tmp/gmsh-install.sh && bash /tmp/gmsh-install.sh')
    os.system("pip install 'viskex[backend-dolfinx]'")


In [2]:
import numpy as np
import ufl
from mpi4py import MPI
import dolfinx
from dolfinx.fem.petsc import LinearProblem
import gmsh
import viskex
from viskex.dolfinx_plotter import DolfinxPlotter
from viskex.dolfinx_converter import DolfinxConverter
import pyvista
import matplotlib.pyplot as plt

## Problem setup

Choose the body here: `"circle"` or `"egg"`. We will start with the circle first, as it is simpler. 
Don't mess with the mesh generation functions - the values I am going to ask you to report depend greatly on it. 

In [3]:
body = "circle"     # "circle" or "egg"
level = 1           # mesh refinement: each level splits every triangle into 4

radius = 1.0        # larger radius of body
L = 20.0            # square [-L, L]^2
U = 1.0             # speed of the uniform stream
rho = 1.0           # density of the fluid

# labels for the two parts of the boundary, and for the fluid itself
OUTER, BODY, FLUID = 1, 2, 3

## The mesh
We will simulate flow past a body in a 2-D domain. This mimics flow past a cylinder 
in real 3-D space but is computationally much faster.

We will create a square with the body cut out of it. We assume that
the body is immersed in a much larger fluid body: that is, the square is NOT
the boundary of the physical domain, but is a slice out of a larger fluid. 
This means we are not modeling e.g. pipe flow with some slip or no-slip boundary. 
This choice will inform the boundary conditions to be free-stream on the edges.

Each piece of the boundary gets a label (a *tag*). This is how we will tell FEniCSx
which boundary is which.

We make a square, make the body, and cut one out of the other. Triangles are small
next to the body and large out at the square. To refine, gmsh splits every triangle
into 4, `level` times.

The second body is an egg, tilted to remove symmetry

$$ x(t) = r_1\cos t, \qquad y(t) = r_2\sin t\,(1 - \varepsilon\cos t), $$

In [ ]:
def mesh_circle(r=1.0, L=20.0, level=0):
    """The square [-L, L]^2 with a circle of radius r cut out of the middle."""
    gmsh.initialize()
    gmsh.option.setNumber("General.Terminal", 0)     # stop gmsh printing everything

    square = gmsh.model.occ.addRectangle(-L, -L, 0, 2*L, 2*L)
    circle = gmsh.model.occ.addDisk(0, 0, 0, r, r)
    gmsh.model.occ.cut([(2, square)], [(2, circle)])

    return label_and_mesh(r, level)


def mesh_egg(r1=1.0, r2 = 0.7,L=20.0, level=0):
    """The square [-L, L]^2 with a tilted egg cut out of the middle."""
    gmsh.initialize()
    gmsh.option.setNumber("General.Terminal", 0)

    eps=0.3; tilt=20
    t = np.linspace(0, 2*np.pi, 50, endpoint=False)
    x = r1*np.cos(t)
    y = r2*np.sin(t)*(1 - eps*np.cos(t))
    angle = np.radians(tilt)
    x, y = x*np.cos(angle) - y*np.sin(angle), x*np.sin(angle) + y*np.cos(angle)

    # define curve through the points
    points = [gmsh.model.occ.addPoint(x[i], y[i], 0) for i in range(len(t))]
    outline = gmsh.model.occ.addSpline(points + [points[0]])
    egg = gmsh.model.occ.addPlaneSurface([gmsh.model.occ.addCurveLoop([outline])])

    square = gmsh.model.occ.addRectangle(-L, -L, 0, 2*L, 2*L)
    gmsh.model.occ.cut([(2, square)], [(2, egg)])

    return label_and_mesh(r1, level)


def label_and_mesh(r, level):
    """Label the boundaries, then make the mesh. Object should have radius <= r."""
    gmsh.model.occ.synchronize()

    # Any curve within 2r of the middle is part of the body
    def near_body(dim):
        # dim = 0 means search for points (as opposed to lines or areas) within 2r of the middle
        return gmsh.model.getEntitiesInBoundingBox(-2*r, -2*r, -1, 2*r, 2*r, 1, dim)

    # labels, so that FEniCSx can tell the boundaries apart
    body_curves = [tag for dim, tag in near_body(1)]
    outer_curves = [tag for dim, tag in gmsh.model.getEntities(1) if tag not in body_curves]
    gmsh.model.addPhysicalGroup(1, outer_curves, OUTER) #The 1 here stands for 1-dimensional, i.e. a line
    gmsh.model.addPhysicalGroup(1, body_curves, BODY) # We are searching for line objects that BCs can apply to 
    gmsh.model.addPhysicalGroup(2, [tag for dim, tag in gmsh.model.getEntities(2)], FLUID) #The fluid is 2-dimensional 

    # triangle size: 0.4 at the body, 4 at the wall edges, blended in between
    gmsh.model.mesh.setSize(gmsh.model.getEntities(0), 4.0)
    gmsh.model.mesh.setSize(near_body(0), 0.4)

    gmsh.model.mesh.generate(2)
    for i in range(level):
        gmsh.model.mesh.refine()      # split every triangle into 4
    gmsh.model.mesh.setOrder(2)       # curved triangle edges, so the circle is actually round

    domain, subdomains, facet_tags, *_ = dolfinx.io.gmsh.model_to_mesh(
        gmsh.model, MPI.COMM_WORLD, 0, gdim=2)
    gmsh.finalize()
    return domain, facet_tags

In [ ]:
if body == "circle":
    domain, facet_tags = mesh_circle(r=radius, L=L, level=level)
else:
    domain, facet_tags = mesh_egg(r1=radius, L=L, level=level)

viskex.dolfinx.plot_mesh(domain)
viskex.dolfinx.plot_mesh_tags(domain, facet_tags, "boundaries")

## The solver

This homework has **four blanks**, marked `YOUR CODE HERE`: blank 1 in `uniform_stream`,
blanks 2 and 3 in `solve_potential`, and blank 4 in `drag`. Fill in each one and delete
the `raise` line beneath its comment.

Multiply $\Delta\phi = 0$ by a test function $v$ and integrate by parts:

$$
\int_{\Omega} \nabla \cdot \nabla \phi \cdot v = - \int_\Omega \nabla \phi \cdot \nabla v + \int_{\partial \Omega} v \nabla \phi \cdot n
$$

Some technical matters: we assume that, far from the circle, the fluid velocity follows 
a prescribed stream with a given velocity. The boundaries of the square should then
be prescribed to match the free-stream velocity. When doing this variationally, we assume that our test function is 0 on this boundary. 
This is known as a Dirichlet boundary condition.

On the body, we cannot prescribe a Dirichlet boundary condition because we do not know the fluid velocity on the object
(this is the main thing we are trying to find). Instead, we impose a Neumann boundary
condition by requiring the normal derivative of $\phi$ to vanish on the body. 
That is, $\nabla \phi \cdot n = 0 \iff u \cdot n = 0$.
This is the no-penetration boundary (fluid cannot enter the body).

With these specifications, the boundary term vanishes and the weak form is
$\int_\Omega \nabla\phi \cdot \nabla v\,dx = 0$.

In [ ]:
def uniform_stream(x):
    """Far from the body, phi = U x."""
    # YOUR CODE HERE (blank 1): return the far-field potential.
    # x is a 3 by N array of coordinates: its columns hold the x, y and z coordinates of N points.
    raise NotImplementedError("Blank 1: fill in uniform_stream")

In [ ]:
def solve_potential(domain, facet_tags, boundary_func):
    """Solve Δφ = 0 with φ = boundary_func on the outer boundary. Nothing on the body."""
    V = dolfinx.fem.functionspace(domain, ("Lagrange", 2))   # quadratic this time

    # boundary condition: phi = boundary_func on the outer boundary only
    u_bc = dolfinx.fem.Function(V)
    u_bc.interpolate(boundary_func)
    domain.topology.create_connectivity(1, 2)
    # YOUR CODE HERE (blank 2): find the degrees of freedom where the Dirichlet condition applies.
    # The Poisson solver used every exterior facet. What would that do to the body?
    # Hint: facet_tags.find(tag) gives the facets with a given tag.
    # Hint 2: boundary_dofs = dolfinx.fem.locate_dofs_topological(...) is going to be helpful.
    # Look up the example in https://jsdokken.com/dolfinx-tutorial/stable/chapter2/heat_code.html
    raise NotImplementedError("Blank 2: fill in boundary_dofs in solve_potential")
    boundary_dofs = ...
    bc = dolfinx.fem.dirichletbc(u_bc, boundary_dofs)

    # weak form:  ∫ ∇φ·∇v dx = 0
    phi, v = ufl.TrialFunction(V), ufl.TestFunction(V)
    zero = dolfinx.fem.Constant(domain, 0.0)     # FEniCSx needs a zero it can integrate
    # YOUR CODE HERE (blank 3): write the left side of the weak form above in UFL.
    raise NotImplementedError("Blank 3: fill in the weak form in solve_potential")
    a = ...
    rhs = zero*v*ufl.dx

    problem = LinearProblem(a, rhs, bcs=[bc], petsc_options_prefix="potential_",
                            petsc_options={"ksp_type": "preonly", "pc_type": "lu"})
    return problem.solve()

In [ ]:
phi = solve_potential(domain, facet_tags, uniform_stream)

## Plotting flows

The velocity is $\nabla\phi$. The colour is the speed and the white lines are
streamlines, the paths the fluid follows.

In [ ]:
def plot_flow(phi):
    # velocity = gradient of phi
    W = dolfinx.fem.functionspace(phi.function_space.mesh, ("Lagrange", 1, (2,)))
    velocity = dolfinx.fem.Function(W)
    velocity.interpolate(dolfinx.fem.Expression(ufl.grad(phi), W.element.interpolation_points))

    # start the streamlines on a line upstream of the body
    ys = np.linspace(-3, 3, 31)
    seeds = pyvista.PolyData(np.column_stack([np.full_like(ys, -6), ys, 0*ys]))

    plotter = DolfinxPlotter.plot_vector_field(velocity, "velocity", show_edges=False, cmap="viridis",
                                               plotter=pyvista.Plotter(window_size=(700, 650)))
    grid = DolfinxConverter.convert_field(velocity, "velocity")
    lines = grid.streamlines_from_source(seeds, vectors="velocity", surface_streamlines=True,
                                         max_length=20.0, initial_step_length=0.05, max_steps=5000)
    plotter.add_mesh(lines, color="white", line_width=1.5)
    plotter.view_xy()
    plotter.camera.zoom(4)     # close up on the body
    plotter.show()

In [ ]:
plot_flow(phi)

- Change `body` to `"egg"` at the top and run everything again. What changes?
- The Poisson solver from previous demos put the Dirichlet condition on *every* boundary. Try that here
  (`dolfinx.mesh.exterior_facet_indices(domain.topology)`). What happens to the flow, and why?

## Drag

**Force.** In the last homework we considered the forces on a body immersed in an ideal fluid. 
$$
\mathbf{F} = \int_{\Gamma} p \mathbf{n}\,ds
$$
Where $\Gamma$ is the fluid-body boundary. Furthermore, you showed that this can be calculated 
using an application of Bernoulli's Theorem and the Euler Momentum equation as,

$$ \mathbf{F} = -\frac{\rho}{2}\oint (\mathbf{u}\cdot\mathbf{u})\,\mathbf{n}\,ds , $$

In both formulas $\mathbf{n}$ points out of the fluid, into the body. Conveniently, that is
the direction of `ufl.FacetNormal`.

This is the part of the calculation that depends on the Euler equations. The Euler equations
are the limit of the NSE for high Reynolds numbers. In particular, given that we have specified most terms
of the Reynolds number, you can specifically identify which Reynolds number component must approach a limiting value 
in this demo. 

**Drag** is the part of the force pointing downstream, along the unit vector
$\hat{\mathbf{e}}$ in the direction of the flow. **Lift** is the part perpendicular to the flow.
Both are scaled by the body width $2r$:

$$ C_D = \frac{2 \mathbf{F}\cdot\hat{\mathbf{e}}}{\rho U^2\,(2r)}, \qquad
   C_L = \frac{2 \mathbf{F}\cdot\hat{\mathbf{e}}_\perp}{\rho U^2\,(2r)} . $$
The choice of 2r is somewhat arbitrary in the case of the egg, but for this test the scaling does not really matter. 

In [ ]:
def drag(phi, facet_tags):
    """Drag and lift coefficients on the body."""
    domain = phi.function_space.mesh
    n = ufl.FacetNormal(domain)              # points out of the fluid, into the body
    ds_body = ufl.Measure("ds", domain=domain, subdomain_data=facet_tags, subdomain_id=BODY)
    velocity = ufl.grad(phi)

    # the force on the body
    # YOUR CODE HERE (blank 4): the two components of the force formula above.
    # dolfinx.fem.assemble_scalar(dolfinx.fem.form(...)) turns a UFL integral into a number.
    raise NotImplementedError("Blank 4: fill in Fx and Fy in drag")
    Fx = ...
    Fy = ...
    F = np.array([Fx, Fy])

    # drag: the force along the flow. lift: the force across it.
    downstream = np.array([1.0, 0.0])        # the flow goes in the +x direction
    across = np.array([0.0, 1.0])
    CD = 2*np.dot(F, downstream)/(rho*U**2*2*radius)
    CL = 2*np.dot(F, across)/(rho*U**2*2*radius)
    return CD, CL


CD, CL = drag(phi, facet_tags)
print("CD =", CD)
print("CL =", CL)

Is that physically realistic? Look at the following wikipedia article which includes
some drag coefficients for physical objects: https://en.wikipedia.org/wiki/Drag_coefficient
To check if this is an issue with our solver, we can refine the mesh and see how the drag changes.

The mesh is not the only source of error. Setting $\phi = Ux$ on the outer boundary
pretends the body has no effect on the flow at distance $L$, which is only true
infinitely far away. That is why the square is 10 diameters wide: far enough that
this error is much smaller than the ones we will see here.

In [ ]:
levels = [0, 1, 2, 3, 4] # refinement level
drags = []

print("level   triangles      CD          CL")
for level in levels:
    if body == "circle":
        domain, facet_tags = mesh_circle(r=radius, L=L, level=level)
    else:
        domain, facet_tags = mesh_egg(r1=radius, L=L, level=level)
    phi = solve_potential(domain, facet_tags, uniform_stream)
    CD, CL = drag(phi, facet_tags)
    drags.append(abs(CD))
    print(f"{level:5d} {domain.topology.index_map(2).size_local:10d}   {CD:10.2e}  {CL:10.2e}")

In [ ]:
plt.semilogy(levels, drags, "o-")
plt.xlabel("refinement level")
plt.ylabel("|CD|")
plt.title(f"Drag on the {body}")
plt.show()

*"I must confess that I do not know how the resistance of fluids can be explained
 theoretically, because the theory gives zero resistance..."* -d'Alembert, 1768.

- Try the egg. Is it the symmetry of the circle that makes the drag vanish?
- Real cylinders have $C_D \approx 1$. What have we left out?